In [ ]:
import pandas as pd

house_sales = pd.read_csv("house_sales.csv")

missing_city = (house_sales["city"] == "--").sum()

missing_city

In [ ]:
clean_data = house_sales.copy()

clean_data["city"] = clean_data["city"].replace("--", "Unknown")

clean_data = clean_data.dropna(subset=["sale_price"])

clean_data["sale_date"] = clean_data["sale_date"].fillna("2023-01-01")

months_mean = round(clean_data["months_listed"].mean(), 1)
clean_data["months_listed"] = clean_data["months_listed"].fillna(months_mean)

bedrooms_mean = round(clean_data["bedrooms"].mean())
clean_data["bedrooms"] = clean_data["bedrooms"].fillna(bedrooms_mean).astype(int)

clean_data["house_type"] = clean_data["house_type"].replace({
    "Terr.": "Terraced",
    "Semi": "Semi-detached",
    "Det.": "Detached"
})

most_common_type = clean_data["house_type"].mode()[0]
clean_data["house_type"] = clean_data["house_type"].fillna(most_common_type)

clean_data["area"] = (
    clean_data["area"]
    .astype(str)
    .str.replace(" sq.m.", "", regex=False)
)

clean_data["area"] = pd.to_numeric(clean_data["area"], errors="coerce")

area_mean = round(clean_data["area"].mean(), 1)
clean_data["area"] = clean_data["area"].fillna(area_mean).round(1)

clean_data["sale_price"] = clean_data["sale_price"].astype(int)

clean_data

In [ ]:
price_by_rooms = (
    house_sales
    .groupby("bedrooms")["sale_price"]
    .agg(["mean", "var"])
    .reset_index()
)

price_by_rooms = price_by_rooms.rename(columns={
    "mean": "avg_price",
    "var": "var_price"
})

price_by_rooms["avg_price"] = price_by_rooms["avg_price"].round(1)
price_by_rooms["var_price"] = price_by_rooms["var_price"].round(1)

price_by_rooms

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

train = pd.read_csv("train.csv")
validation = pd.read_csv("validation.csv")

y_train = train["sale_price"]

X_train = train.drop(columns=["house_id", "sale_price"]).copy()
X_validation = validation.drop(columns=["house_id"]).copy()

X_train["sale_date"] = pd.to_datetime(X_train["sale_date"])
X_validation["sale_date"] = pd.to_datetime(X_validation["sale_date"])

for df in [X_train, X_validation]:
    df["year"] = df["sale_date"].dt.year
    df["month"] = df["sale_date"].dt.month
    df["day"] = df["sale_date"].dt.day
    df.drop(columns=["sale_date"], inplace=True)

categorical_features = ["city", "house_type"]

numerical_features = [
    "months_listed",
    "bedrooms",
    "area",
    "year",
    "month",
    "day"
]

preprocessor = ColumnTransformer([
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_features
    ),
    (
        "numerical",
        SimpleImputer(strategy="median"),
        numerical_features
    )
])

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_validation)

base_result = pd.DataFrame({
    "house_id": validation["house_id"],
    "price": baseline_predictions.round(1)
})

base_result

In [ ]:
from sklearn.ensemble import RandomForestRegressor

train = pd.read_csv("train.csv")
validation = pd.read_csv("validation.csv")

y_train = train["sale_price"]

X_train = train.drop(columns=["house_id", "sale_price"]).copy()
X_validation = validation.drop(columns=["house_id"]).copy()

X_train["sale_date"] = pd.to_datetime(X_train["sale_date"])
X_validation["sale_date"] = pd.to_datetime(X_validation["sale_date"])

for df in [X_train, X_validation]:
    df["year"] = df["sale_date"].dt.year
    df["month"] = df["sale_date"].dt.month
    df["day"] = df["sale_date"].dt.day
    df.drop(columns=["sale_date"], inplace=True)

categorical_features = ["city", "house_type"]

numerical_features = [
    "months_listed",
    "bedrooms",
    "area",
    "year",
    "month",
    "day"
]

preprocessor = ColumnTransformer([
    (
        "categorical",
        OneHotEncoder(handle_unknown="ignore"),
        categorical_features
    ),
    (
        "numerical",
        SimpleImputer(strategy="median"),
        numerical_features
    )
])

comparison_model = Pipeline([
    ("preprocessor", preprocessor),
    (
        "model",
        RandomForestRegressor(
            n_estimators=500,
            random_state=42,
            n_jobs=-1
        )
    )
])

comparison_model.fit(X_train, y_train)

comparison_predictions = comparison_model.predict(X_validation)

compare_result = pd.DataFrame({
    "house_id": validation["house_id"],
    "price": comparison_predictions.round(1)
})

compare_result

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Split the training data into training and test sets
X_train_rmse, X_test_rmse, y_train_rmse, y_test_rmse = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42
)

# Baseline model
baseline_model.fit(X_train_rmse, y_train_rmse)
baseline_test_predictions = baseline_model.predict(X_test_rmse)

base_rmse = np.sqrt(
    mean_squared_error(y_test_rmse, baseline_test_predictions)
)

# Comparison model
comparison_model.fit(X_train_rmse, y_train_rmse)
comparison_test_predictions = comparison_model.predict(X_test_rmse)

compare_rmse = np.sqrt(
    mean_squared_error(y_test_rmse, comparison_test_predictions)
)

print("Baseline RMSE:", base_rmse)
print("Comparison RMSE:", compare_rmse)